# Mined vs Controller Trace Replay Metrics

Analyze validation mined-trace replay outputs alongside controller-generated traces and the new controller-trace replay outputs produced by `run_scripts/run_m3cot_validation_ablation_entropy_eval.sh`.

The notebook is intentionally discovery-based: it works from a local checkout, or from the server output root at `/home/csalt/Haider/DVLM/lvar`, when those artifacts are mounted.

In [ ]:
%matplotlib inline

from collections import Counter
import json
from pathlib import Path
import re

from IPython.display import display
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    sns = None

PROJECT_ROOT = Path.cwd() if Path.cwd().name != "analysis" else Path.cwd().parent
SERVER_ROOT = Path("/home/csalt/Haider/DVLM/lvar")

DATASET_PARTITION = "validation"
LOCAL_OUTPUT_ROOT = PROJECT_ROOT / "outputs"
SERVER_OUTPUT_ROOT = SERVER_ROOT / "outputs"
OUTPUT_ROOTS = [root for root in [LOCAL_OUTPUT_ROOT, SERVER_OUTPUT_ROOT] if root.exists()]

ABLATION_ROOTS = [root / "inference" / f"{DATASET_PARTITION}_oracle_ablations" for root in OUTPUT_ROOTS]
SWEEP_ROOTS = [root / "controller_sft_m3cot_test_sweeps" for root in OUTPUT_ROOTS]
ORACLE_TRACE_ROOTS = [root / "oracle_dataset" / DATASET_PARTITION / "lvar_ckpt" for root in OUTPUT_ROOTS]

print("Project root:", PROJECT_ROOT)
print("Output roots:", OUTPUT_ROOTS)
print("Ablation roots:", [p for p in ABLATION_ROOTS if p.exists()])
print("Sweep roots:", [p for p in SWEEP_ROOTS if p.exists()])

## Helpers

In [ ]:
def read_json(path: Path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def read_jsonl(path: Path):
    rows = []
    bad = []
    with Path(path).open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            stripped = line.strip()
            if not stripped:
                continue
            try:
                rows.append(json.loads(stripped))
            except json.JSONDecodeError as exc:
                bad.append((line_number, str(exc)))
    if bad:
        print(f"Warning: {path} had {len(bad)} bad JSONL rows; first={bad[0]}")
    return rows


def maybe_read_json(path: Path):
    return read_json(path) if Path(path).exists() else None


def entropy_path_for_prediction(prediction_path: Path) -> Path:
    return prediction_path.with_name(f"{prediction_path.stem}_entropy_tracking.json")


def summary_path_for_prediction(prediction_path: Path) -> Path:
    return prediction_path.with_name(f"{prediction_path.stem}_summary.json")


def action_name(step):
    return str(step.get("action") or step.get("type") or "").upper()


def trace_actions(row):
    return [action_name(step) for step in (row.get("trace") or [])]


def trace_stats(row):
    actions = trace_actions(row)
    counts = Counter(actions)
    return {
        "num_steps": len(actions),
        "num_think": counts.get("THINK", 0),
        "num_global": counts.get("GLOBAL", 0),
        "num_region": counts.get("REGION", 0),
        "num_patch": counts.get("PATCH", 0),
        "num_stop": counts.get("STOP", 0),
        "num_visual": counts.get("GLOBAL", 0) + counts.get("REGION", 0) + counts.get("PATCH", 0),
        "action_sequence": " ".join(actions),
    }


def levenshtein(a, b):
    previous = list(range(len(b) + 1))
    for i, token_a in enumerate(a, start=1):
        current = [i]
        for j, token_b in enumerate(b, start=1):
            current.append(min(
                previous[j] + 1,
                current[j - 1] + 1,
                previous[j - 1] + (token_a != token_b),
            ))
        previous = current
    return previous[-1]


def safe_auc(df, score_col, label_col="correct"):
    usable = df[[score_col, label_col]].dropna().copy()
    if usable.empty or usable[label_col].nunique() < 2:
        return np.nan
    y_error = (~usable[label_col].astype(bool)).astype(int).to_numpy()
    scores = usable[score_col].astype(float).to_numpy()
    order = np.argsort(scores)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(scores) + 1)
    positives = y_error == 1
    n_pos = positives.sum()
    n_neg = len(scores) - n_pos
    if n_pos == 0 or n_neg == 0:
        return np.nan
    return float((ranks[positives].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


def display_or_empty(df, columns=None, n=20):
    if df.empty:
        print("No rows found.")
        return
    display(df[columns].head(n) if columns else df.head(n))

## Discover Runs

This loads three families of artifacts:

- mined oracle traces from `outputs/oracle_dataset/validation/lvar_ckpt`
- direct controller validation predictions from `outputs/controller_sft_m3cot_test_sweeps/eval_validation/...`
- replayed controller traces from `outputs/inference/validation_oracle_ablations/controller_traces_from_lvar_model/...`

In [ ]:
def load_prediction_run(prediction_path: Path, run_family: str, trace_source=None, variant=None, replay_kind=None):
    prediction_rows = read_jsonl(prediction_path)
    summary = maybe_read_json(summary_path_for_prediction(prediction_path)) or {}
    entropy_path = Path(summary.get("entropy_tracking_path") or entropy_path_for_prediction(prediction_path))
    entropy_rows = read_json(entropy_path) if entropy_path.exists() else []

    row_records = []
    for row in prediction_rows:
        record = {
            "run_family": run_family,
            "trace_source": trace_source,
            "variant": variant,
            "replay_kind": replay_kind,
            "prediction_path": str(prediction_path),
            "summary_path": str(summary_path_for_prediction(prediction_path)) if summary_path_for_prediction(prediction_path).exists() else None,
            "entropy_path": str(entropy_path) if entropy_path.exists() else None,
            "example_id": str(row.get("example_id")),
            "correct": bool(row.get("correct")),
            "domain": row.get("domain"),
            "topic": row.get("topic"),
            "gold_answer": row.get("gold_answer"),
            "decoded_answer": row.get("decoded_answer") or row.get("answer"),
            "generated_text": row.get("generated_text"),
            "num_output_tokens": row.get("num_output_tokens"),
            "trace": row.get("trace") or [],
            "action_prefix_metrics": row.get("action_prefix_metrics") or [],
        }
        record.update(trace_stats(row))
        row_records.append(record)

    entropy_records = []
    for row in entropy_rows:
        record = {
            "run_family": run_family,
            "trace_source": trace_source,
            "variant": variant,
            "replay_kind": replay_kind,
            "prediction_path": str(prediction_path),
            "entropy_path": str(entropy_path),
            "example_id": str(row.get("example_id")),
            "correct": bool(row.get("correct")),
            "gold_answer": row.get("gold_answer"),
            "decoded_answer": row.get("decoded_answer"),
            "num_output_tokens": row.get("num_output_tokens"),
            "answer_option_entropy": row.get("answer_option_entropy"),
            "decoded_token_entropy_mean": row.get("decoded_token_entropy_mean"),
            "controller_action_entropy_mean": row.get("controller_action_entropy_mean"),
            "controller_region_entropy_mean": row.get("controller_region_entropy_mean"),
            "controller_patch_entropy_mean": row.get("controller_patch_entropy_mean"),
            "controller_entropy_mean": row.get("controller_entropy_mean"),
            "controller_entropies": row.get("controller_entropies") or [],
            "controller_action_entropies": row.get("controller_action_entropies") or [],
            "controller_region_entropies": row.get("controller_region_entropies") or [],
            "controller_patch_entropies": row.get("controller_patch_entropies") or [],
        }
        entropy_records.append(record)

    metrics = dict(summary.get("metrics", {}))
    summary_record = {
        "run_family": run_family,
        "trace_source": trace_source,
        "variant": variant,
        "replay_kind": replay_kind,
        "prediction_path": str(prediction_path),
        "entropy_path": str(entropy_path) if entropy_path.exists() else None,
        "total": metrics.get("total") or metrics.get("num_examples") or len(row_records),
        "accuracy": metrics.get("accuracy"),
        "avg_trace_actions": metrics.get("avg_trace_actions") or metrics.get("avg_controller_actions"),
        "avg_output_tokens": metrics.get("avg_output_tokens"),
        "step_hidden_entropy_mean": metrics.get("step_hidden_entropy_mean"),
        "step_hidden_entropy_count": metrics.get("step_hidden_entropy_count"),
        "prefix_rollout_accuracy": (metrics.get("prefix_rollouts") or {}).get("accuracy"),
        "prefix_rollout_total": (metrics.get("prefix_rollouts") or {}).get("total"),
    }
    if summary_record["accuracy"] is None and row_records:
        summary_record["accuracy"] = float(np.mean([r["correct"] for r in row_records]))
    return row_records, entropy_records, summary_record


prediction_records = []
entropy_records = []
summary_records = []

# Mined raw validation replay from the ablation entropy run.
for root in ABLATION_ROOTS:
    pattern = "mined_by_lvar_ckpt/evaluated_by_lvar_ckpt/trace_variant_raw/m3cot_validation_predictions_mined-by_lvar_evaluated-by_lvar_*_raw*.jsonl"
    for path in sorted(root.glob(pattern)):
        trace_context = "coarse" if "_coarse_" in path.name else "global"
        rows, entropy, summary = load_prediction_run(path, "mined_replay", "lvar_ckpt", "raw", trace_context)
        prediction_records.extend(rows)
        entropy_records.extend(entropy)
        summary_records.append(summary)

# Controller-trace replay outputs added by run_m3cot_validation_ablation_entropy_eval.sh.
for root in ABLATION_ROOTS:
    base = root / "controller_traces_from_lvar_model"
    for path in sorted(base.glob("*/*/m3cot_validation_predictions_replayed-controller-traces_*.jsonl")):
        trace_source = path.parent.parent.name
        variant = path.parent.name
        rows, entropy, summary = load_prediction_run(path, "controller_trace_replay", trace_source, variant, "forced_replay")
        prediction_records.extend(rows)
        entropy_records.extend(entropy)
        summary_records.append(summary)

# Direct controller validation traces from the sweep, before forced replay.
for sweep_root in SWEEP_ROOTS:
    base = sweep_root / "eval_validation"
    for path in sorted(base.glob("*/*/m3cot_validation_predictions.jsonl")):
        trace_source = path.parent.parent.name
        variant = path.parent.name
        rows, entropy, summary = load_prediction_run(path, "controller_direct", trace_source, variant, "model_generated")
        prediction_records.extend(rows)
        entropy_records.extend(entropy)
        summary_records.append(summary)

pred_df = pd.DataFrame(prediction_records)
entropy_df = pd.DataFrame(entropy_records)
summary_df = pd.DataFrame(summary_records).drop_duplicates(subset=["prediction_path"])

print(f"Prediction rows: {len(pred_df):,}")
print(f"Entropy rows: {len(entropy_df):,}")
print(f"Runs: {len(summary_df):,}")
if summary_df.empty:
    print("No prediction runs found yet.")
else:
    display(summary_df.sort_values(["run_family", "trace_source", "variant", "replay_kind"]))

## Mined Trace Inventory

This section summarizes the raw mined oracle traces themselves, independent of replay output.

In [ ]:
mined_trace_records = []
for root in ORACLE_TRACE_ROOTS:
    for path in sorted(root.glob("m3cot_validation_traces_lvar_*.jsonl")):
        context = "coarse" if path.name.endswith("_coarse.jsonl") else "global"
        for row in read_jsonl(path):
            record = {
                "trace_path": str(path),
                "context": context,
                "example_id": str(row.get("example_id")),
                "num_decisions": len(row.get("decisions") or []),
            }
            record.update(trace_stats(row))
            mined_trace_records.append(record)

mined_trace_df = pd.DataFrame(mined_trace_records)
if mined_trace_df.empty:
    print("No mined validation trace files found.")
else:
    mined_summary = mined_trace_df.groupby("context").agg(
        examples=("example_id", "nunique"),
        mean_steps=("num_steps", "mean"),
        mean_decisions=("num_decisions", "mean"),
        mean_visual=("num_visual", "mean"),
        mean_think=("num_think", "mean"),
        mean_region=("num_region", "mean"),
        mean_patch=("num_patch", "mean"),
    ).reset_index()
    display(mined_summary.style.format({c: "{:.2f}" for c in mined_summary.columns if c.startswith("mean_")}))

## Run-Level Metrics

In [ ]:
if summary_df.empty:
    print("No run summaries found yet.")
else:
    view = summary_df.copy()
    view["accuracy_pct"] = 100 * pd.to_numeric(view["accuracy"], errors="coerce")
    cols = ["run_family", "trace_source", "variant", "replay_kind", "total", "accuracy", "avg_trace_actions", "avg_output_tokens", "step_hidden_entropy_mean", "prefix_rollout_accuracy", "prefix_rollout_total"]
    display(view[cols].sort_values(["run_family", "trace_source", "variant"]).style.format({
        "accuracy": "{:.2%}",
        "avg_trace_actions": "{:.2f}",
        "avg_output_tokens": "{:.2f}",
        "step_hidden_entropy_mean": "{:.4f}",
        "prefix_rollout_accuracy": "{:.2%}",
    }))

    plt.figure(figsize=(10, 4))
    if sns is not None:
        sns.barplot(data=view, x="variant", y="accuracy_pct", hue="run_family")
    else:
        view.plot.bar(x="variant", y="accuracy_pct", ax=plt.gca())
    plt.ylabel("Accuracy")
    plt.gca().yaxis.set_major_formatter(PercentFormatter(100))
    plt.xticks(rotation=30, ha="right")
    plt.title("Validation accuracy by trace source and replay family")
    plt.tight_layout()

## Trace Shape And Action Mix

In [ ]:
if pred_df.empty:
    print("No predictions loaded.")
else:
    trace_summary = pred_df.groupby(["run_family", "trace_source", "variant", "replay_kind"], dropna=False).agg(
        examples=("example_id", "nunique"),
        accuracy=("correct", "mean"),
        mean_steps=("num_steps", "mean"),
        mean_visual=("num_visual", "mean"),
        mean_think=("num_think", "mean"),
        mean_region=("num_region", "mean"),
        mean_patch=("num_patch", "mean"),
        mean_output_tokens=("num_output_tokens", "mean"),
    ).reset_index()
    display(trace_summary.style.format({
        "accuracy": "{:.2%}",
        "mean_steps": "{:.2f}",
        "mean_visual": "{:.2f}",
        "mean_think": "{:.2f}",
        "mean_region": "{:.2f}",
        "mean_patch": "{:.2f}",
        "mean_output_tokens": "{:.2f}",
    }))

    long_actions = pred_df.melt(
        id_vars=["run_family", "trace_source", "variant", "replay_kind", "example_id", "correct"],
        value_vars=["num_think", "num_global", "num_region", "num_patch", "num_stop"],
        var_name="action",
        value_name="count",
    )
    long_actions["action"] = long_actions["action"].str.replace("num_", "", regex=False).str.upper()
    plt.figure(figsize=(11, 4))
    if sns is not None:
        sns.barplot(data=long_actions, x="variant", y="count", hue="action", errorbar=None)
    else:
        long_actions.groupby(["variant", "action"])["count"].mean().unstack().plot.bar(ax=plt.gca())
    plt.ylabel("Mean count per example")
    plt.title("Action mix across loaded runs")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()

## Pair Controller Runs Against Mined Replay

For each controller variant, compare its action sequence to the mined raw global replay on the same example. Edit distance is computed over action names only, not visual indices.

In [ ]:
if pred_df.empty:
    print("No predictions loaded.")
else:
    mined_base = pred_df[(pred_df.run_family == "mined_replay") & (pred_df.replay_kind == "global")].copy()
    if mined_base.empty:
        mined_base = pred_df[pred_df.run_family == "mined_replay"].copy()
    mined_base = mined_base.sort_values("prediction_path").drop_duplicates("example_id")
    controller_like = pred_df[pred_df.run_family.isin(["controller_direct", "controller_trace_replay"])].copy()
    paired = controller_like.merge(
        mined_base[["example_id", "correct", "num_steps", "num_visual", "action_sequence", "trace"]].rename(columns={
            "correct": "mined_correct",
            "num_steps": "mined_num_steps",
            "num_visual": "mined_num_visual",
            "action_sequence": "mined_action_sequence",
            "trace": "mined_trace",
        }),
        on="example_id",
        how="inner",
    )
    if paired.empty:
        print("No overlapping example_ids between mined replay and controller runs.")
    else:
        paired["action_edit_distance"] = [levenshtein(trace_actions({"trace": a}), trace_actions({"trace": b})) for a, b in zip(paired["trace"], paired["mined_trace"])]
        paired["delta_steps_vs_mined"] = paired["num_steps"] - paired["mined_num_steps"]
        paired["delta_visual_vs_mined"] = paired["num_visual"] - paired["mined_num_visual"]
        paired["correctness_change"] = np.select(
            [paired["correct"] & ~paired["mined_correct"], ~paired["correct"] & paired["mined_correct"], paired["correct"] == paired["mined_correct"]],
            ["controller_improves", "controller_regresses", "same"],
            default="other",
        )
        compare = paired.groupby(["run_family", "trace_source", "variant", "replay_kind", "correctness_change"], dropna=False).agg(
            examples=("example_id", "nunique"),
            mean_edit_distance=("action_edit_distance", "mean"),
            mean_delta_steps=("delta_steps_vs_mined", "mean"),
            mean_delta_visual=("delta_visual_vs_mined", "mean"),
        ).reset_index()
        display(compare.style.format({
            "mean_edit_distance": "{:.2f}",
            "mean_delta_steps": "{:+.2f}",
            "mean_delta_visual": "{:+.2f}",
        }))
        display(paired.sort_values("action_edit_distance", ascending=False)[[
            "run_family", "trace_source", "variant", "example_id", "correct", "mined_correct", "action_edit_distance", "delta_steps_vs_mined", "delta_visual_vs_mined", "action_sequence", "mined_action_sequence"
        ]].head(15))

## Entropy And Error Detection

Entropy sidecars contain final-token entropy for all replay runs, plus controller-head entropy for direct controller runs and forced controller-trace replays.

In [ ]:
if entropy_df.empty:
    print("No entropy sidecars loaded.")
else:
    metric_cols = [
        "answer_option_entropy",
        "decoded_token_entropy_mean",
        "controller_action_entropy_mean",
        "controller_region_entropy_mean",
        "controller_patch_entropy_mean",
        "controller_entropy_mean",
    ]
    entropy_summary = entropy_df.groupby(["run_family", "trace_source", "variant", "replay_kind"], dropna=False).agg(
        examples=("example_id", "nunique"),
        accuracy=("correct", "mean"),
        **{f"mean_{col}": (col, "mean") for col in metric_cols},
    ).reset_index()
    display(entropy_summary.style.format({c: "{:.4f}" for c in entropy_summary.columns if c.startswith("mean_")} | {"accuracy": "{:.2%}"}))

    auc_rows = []
    for keys, group in entropy_df.groupby(["run_family", "trace_source", "variant", "replay_kind"], dropna=False):
        for metric in metric_cols:
            auc_rows.append({
                "run_family": keys[0],
                "trace_source": keys[1],
                "variant": keys[2],
                "replay_kind": keys[3],
                "metric": metric,
                "coverage": group[metric].notna().mean() if metric in group else 0.0,
                "error_detection_auc": safe_auc(group, metric) if metric in group else np.nan,
            })
    auc_df = pd.DataFrame(auc_rows)
    display(auc_df.sort_values(["run_family", "trace_source", "variant", "metric"]).style.format({"coverage": "{:.1%}", "error_detection_auc": "{:.3f}"}))

    controller_entropy = entropy_df.dropna(subset=["controller_entropy_mean"]).copy()
    if not controller_entropy.empty:
        plt.figure(figsize=(10, 4))
        if sns is not None:
            sns.boxplot(data=controller_entropy, x="variant", y="controller_entropy_mean", hue="correct", showfliers=False)
        else:
            controller_entropy.boxplot(column="controller_entropy_mean", by="variant", ax=plt.gca())
        plt.ylabel("Mean joint controller entropy")
        plt.title("Controller entropy by correctness")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()

## Step-Level Controller Entropy Trajectories

In [ ]:
step_rows = []
if not entropy_df.empty:
    for _, row in entropy_df.iterrows():
        joint = row.get("controller_entropies") or []
        action = row.get("controller_action_entropies") or []
        region = row.get("controller_region_entropies") or []
        patch = row.get("controller_patch_entropies") or []
        for idx in range(max(map(len, [joint, action, region, patch])) if any([joint, action, region, patch]) else 0):
            step_rows.append({
                "run_family": row.get("run_family"),
                "trace_source": row.get("trace_source"),
                "variant": row.get("variant"),
                "replay_kind": row.get("replay_kind"),
                "example_id": row.get("example_id"),
                "correct": row.get("correct"),
                "step_idx": idx,
                "controller_entropy": joint[idx] if idx < len(joint) else np.nan,
                "action_entropy": action[idx] if idx < len(action) else np.nan,
                "region_entropy": region[idx] if idx < len(region) else np.nan,
                "patch_entropy": patch[idx] if idx < len(patch) else np.nan,
            })

step_df = pd.DataFrame(step_rows)
if step_df.empty:
    print("No step-level controller entropy values found.")
else:
    trajectory = step_df.groupby(["run_family", "variant", "step_idx", "correct"], dropna=False)["controller_entropy"].agg(["count", "mean", "std"]).reset_index()
    display(trajectory.head(30).style.format({"mean": "{:.4f}", "std": "{:.4f}"}))
    plt.figure(figsize=(10, 4))
    if sns is not None:
        sns.lineplot(data=trajectory, x="step_idx", y="mean", hue="variant", style="correct", marker="o")
    else:
        for (variant, correct), group in trajectory.groupby(["variant", "correct"]):
            plt.plot(group["step_idx"], group["mean"], marker="o", label=f"{variant} correct={correct}")
        plt.legend()
    plt.ylabel("Mean joint controller entropy")
    plt.title("Controller entropy trajectory")
    plt.tight_layout()

## Prefix Rollout Metrics

Prefix rollouts are expensive and only appear for raw mined replay plus controller-trace replay runs launched with `--track-prefix-rollouts`.

In [ ]:
prefix_rows = []
if not pred_df.empty:
    for _, row in pred_df.iterrows():
        for prefix in row.get("action_prefix_metrics") or []:
            rollout = prefix.get("rollout") or {}
            next_entropy = prefix.get("next_token_entropy") or {}
            prefix_rows.append({
                "run_family": row.run_family,
                "trace_source": row.trace_source,
                "variant": row.variant,
                "replay_kind": row.replay_kind,
                "example_id": row.example_id,
                "final_correct": row.correct,
                "block_idx": prefix.get("block_idx"),
                "prefix_step": (prefix.get("block_idx") if prefix.get("block_idx") is not None else -1) + 1,
                "action": prefix.get("action"),
                "num_primitive_actions": prefix.get("num_primitive_actions"),
                "sequence_length": prefix.get("sequence_length"),
                "next_token_entropy": next_entropy.get("entropy"),
                "retained_probability_mass": next_entropy.get("retained_probability_mass"),
                "rollout_correct": rollout.get("correct"),
                "rollout_answer": rollout.get("decoded_answer"),
                "rollout_token_entropy_mean": rollout.get("token_entropy_mean"),
                "rollout_answer_option_entropy": (rollout.get("answer_option_entropy") or {}).get("entropy"),
            })

prefix_df = pd.DataFrame(prefix_rows)
if prefix_df.empty:
    print("No prefix rollout metrics found.")
else:
    prefix_summary = prefix_df.groupby(["run_family", "variant", "prefix_step", "action"], dropna=False).agg(
        examples=("example_id", "nunique"),
        rollout_accuracy=("rollout_correct", "mean"),
        final_accuracy=("final_correct", "mean"),
        mean_next_token_entropy=("next_token_entropy", "mean"),
        mean_rollout_token_entropy=("rollout_token_entropy_mean", "mean"),
    ).reset_index()
    display(prefix_summary.head(40).style.format({
        "rollout_accuracy": "{:.2%}",
        "final_accuracy": "{:.2%}",
        "mean_next_token_entropy": "{:.4f}",
        "mean_rollout_token_entropy": "{:.4f}",
    }))
    plt.figure(figsize=(10, 4))
    if sns is not None:
        sns.lineplot(data=prefix_summary, x="prefix_step", y="rollout_accuracy", hue="variant", marker="o")
    else:
        for variant, group in prefix_summary.groupby("variant"):
            plt.plot(group["prefix_step"], group["rollout_accuracy"], marker="o", label=variant)
        plt.legend()
    plt.gca().yaxis.set_major_formatter(PercentFormatter(1.0))
    plt.ylabel("Prefix rollout accuracy")
    plt.title("Answer quality after each replayed decision block")
    plt.tight_layout()

## Diagnostic Examples

In [ ]:
if entropy_df.empty:
    print("No entropy rows available for diagnostics.")
else:
    joined = entropy_df.merge(
        pred_df[["run_family", "trace_source", "variant", "replay_kind", "example_id", "domain", "topic", "generated_text", "action_sequence", "num_steps", "num_visual"]],
        on=["run_family", "trace_source", "variant", "replay_kind", "example_id"],
        how="left",
    )
    diagnostic_cols = ["run_family", "trace_source", "variant", "example_id", "domain", "topic", "correct", "gold_answer", "decoded_answer", "controller_entropy_mean", "answer_option_entropy", "num_steps", "num_visual", "action_sequence"]
    controller_rows = joined.dropna(subset=["controller_entropy_mean"])
    if controller_rows.empty:
        print("No controller entropy diagnostics available.")
    else:
        print("Lowest-entropy mistakes")
        display(controller_rows[~controller_rows.correct].nsmallest(12, "controller_entropy_mean")[diagnostic_cols])
        print("Highest-entropy correct examples")
        display(controller_rows[controller_rows.correct].nlargest(12, "controller_entropy_mean")[diagnostic_cols])